In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_pipeline.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE medical_pipeline.gold.fact_encounters AS
with table1 as (
  select encounter_id ,sum(base_cost) as total_cost ,count(*) as procedure_count 
  from medical_pipeline.silver.procedures
  group by encounter_id
) , table2 as(
  select  * from medical_pipeline.silver.encounters_silver enc left join table1 t1 on  enc.id=t1.encounter_id
) ,table3 as (
  select `id` as encounter_id,
  patient as patient_id,
  payer as payer_id,
  encounter_class ,
  `start`,
  `stop`,
  timediff(hour,start,stop) as duration_hours,
  coalesce(total_cost,0) as total_cost,
  coalesce(procedure_count,0) as procedure_count
  from table2

),table4 as (
  select * , 
  case 
  when procedure_count>24 then 1
  else 0
  end as more_than_24 ,
  case
  when payer_id is not null then 1
  else 0
  end as has_payer_coverage
  from table3

  
) select * from table4;

In [0]:
%sql
select * from medical_pipeline.gold.fact_encounters;